In [0]:
from pyspark.sql.functions import col, current_timestamp, to_json, struct, lit, trim, upper, to_date
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

vendor_name = "here_tech"
layer_name = "silver_conform"

print(f"--- Starting Silver Conformance for: {vendor_name} ---")

# 1. Read from Bronze table
df_bronze = spark.read.table("inlap.bronze.heretech")

# 2. Type casting, normalization, and canonical gold-join columns
df_cleaned = (
    df_bronze
    .withColumn("latitude", col("latitude").cast("double"))
    .withColumn("longitude", col("longitude").cast("double"))
    .withColumn("street_address", trim(col("street_address")))
    .withColumn("city", upper(trim(col("city"))))
    .withColumn("state", upper(trim(col("state"))))
    .withColumn("network_type", upper(trim(col("network_type"))))
    .withColumn("signal_strength_dbm", col("signal_strength_dbm").cast("double"))
    .withColumn("event_timestamp", col("last_updated").cast("timestamp"))
    .withColumn("event_date", to_date(col("last_updated")))
)

# 3. Define Quality Rules for mandatory fields (address, signal, and coordinates)
valid_condition = (
    col("street_address").isNotNull()
    & (col("street_address") != "")
    & col("signal_strength_dbm").isNotNull()
    & col("latitude").isNotNull()
    & col("longitude").isNotNull()
)

df_passed_dq = df_cleaned.filter(valid_condition)
df_dq_quarantine = df_cleaned.filter(~valid_condition).withColumn(
    "failure_reason", lit("missing address, coordinates, or signal reading")
)

# 4. Handle duplicates among the data-quality passed records
window_spec = Window.partitionBy("site_id").orderBy(col("event_timestamp").desc(), col("street_address").asc())
df_with_rn = df_passed_dq.withColumn("row_num", row_number().over(window_spec))

df_valid = df_with_rn.filter(col("row_num") == 1).drop("row_num")
df_dup_quarantine = df_with_rn.filter(col("row_num") > 1).drop("row_num").withColumn(
    "failure_reason", lit("duplicate site_id")
)

# 5. Combine both quality failures and duplicate failures into a single quarantine dataframe
df_all_quarantine = df_dq_quarantine.unionByName(df_dup_quarantine)
df_quarantine_final = (
    df_all_quarantine
    .withColumn("source_vendor", lit(vendor_name))
    .withColumn("quarantine_timestamp", current_timestamp())
    .withColumn("raw_record", to_json(struct([col(c) for c in df_cleaned.columns if c not in {"row_num"}])))
    .select("raw_record", "failure_reason", "source_vendor", "quarantine_timestamp")
)

# 6. Refresh Silver Conformed and append quarantine rows
(
    df_valid.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("inlap.silver.heretech_conformed")
)

(
    df_quarantine_final.write.format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable("inlap.silver.quarantine_records")
)

# 7. Log Audit Metrics
rows_read = df_bronze.count()
rows_passed = df_valid.count()
rows_quarantined = df_quarantine_final.count()

spark.sql(f"""
    INSERT INTO inlap.control.audit_log 
    VALUES (
        '{vendor_name}', 
        '{layer_name}', 
        current_timestamp(), 
        'SUCCESS', 
        {rows_read}, 
        {rows_passed}, 
        {rows_quarantined}
    )
""")

print(f"Conformance complete for {vendor_name}. Read: {rows_read} | Passed: {rows_passed} | Quarantined: {rows_quarantined}")

In [0]:
# df = spark.read.table("inlap.bronze.heretech")
# df=df.drop("_ingestion_timestamp", "source_file", "source_name","last_updated")
# display(df.limit(5))

In [0]:
# inlap.control.audit_logfrom pyspark.sql.functions import col, current_timestamp, to_json, struct, lit, trim

# vendor_name = "here_tech"
# layer_name = "silver_conform"

# print(f"--- Starting Silver Conformance for: {vendor_name} ---")

# # 1. Read from Bronze table
# df_bronze = spark.read.table("inlap.bronze.heretech")

# # 2. Type casting and trimming whitespaces
# df_cleaned = df_bronze.withColumn("latitude", col("latitude").cast("double")) \
#                       .withColumn("longitude", col("longitude").cast("double")) \
#                       .withColumn("street_address", trim(col("street_address"))) \
#                       .withColumn("network_type", trim(col("network_type")))

# # 3. Apply Quality Rules & Separate Clean vs Quarantine
# # Condition for valid rows: Address must not be null/empty AND signal must not be null
# valid_condition = col("street_address").isNotNull() & (col("street_address") != "") & col("signal_strength_dbm").isNotNull()

# df_valid = df_cleaned.filter(valid_condition).dropDuplicates()

# # Rows that fail the criteria go to quarantine
# df_quarantine = df_cleaned.filter(~valid_condition) \
#                           .withColumn("failure_reason", lit("missing address or missing signal reading")) \
#                           .withColumn("source_vendor", lit(vendor_name)) \
#                           .withColumn("quarantine_timestamp", current_timestamp()) \
#                           .withColumn("raw_record", to_json(struct([col(c) for c in df_cleaned.columns])))

# # Select only necessary columns for quarantine storage
# df_quarantine_final = df_quarantine.select("raw_record", "failure_reason", "source_vendor", "quarantine_timestamp")

# # 4. Write Valid Records to Silver Conformed Table
# df_valid.write.format("delta") \
#     .mode("append") \
#     .option("mergeSchema", "true") \
#     .saveAsTable("inlap.silver.heretech_conformed")

# # 5. Write Invalid Records to Central Quarantine Table
# df_quarantine_final.write.format("delta") \
#     .mode("append") \
#     .option("mergeSchema", "true") \
#     .saveAsTable("inlap.silver.quarantine_records")

# # 6. Log Audit Metrics
# rows_read = df_bronze.count()
# rows_passed = df_valid.count()
# rows_quarantined = df_quarantine_final.count()

# spark.sql(f"""
#     INSERT INTO inlap.control.audit_log 
#     VALUES ('{vendor_name}', '{layer_name}', current_timestamp(), 'SUCCESS')
# """)

# print(f"Conformance complete for {vendor_name}. Read: {rows_read}, Passed: {rows_passed}, Quarantined: {rows_quarantined}")